## Lab 8 - Ant Colony Optimization (Ant System variant) for TSP

### 1) Goal
Implement the **Ant System (AS)** variant of Ant Colony Optimization (Dorigo, 1992) and use it to solve the Traveling Salesman Problem.
- Check what happens in one iteration for 10 ants
- Determine the global best after 10 iterations
- Test the algorithm with different parameter settings

### 2) Ant System in a nutshell
Each iteration, `m` ants each build a complete tour. At city $i$, an ant chooses the next unvisited city $j$ with probability

$$p_{ij} = \frac{[\tau_{ij}]^\alpha \, [\eta_{ij}]^\beta}{\sum_{k \in \text{allowed}} [\tau_{ik}]^\alpha \, [\eta_{ik}]^\beta}$$

where $\tau_{ij}$ is the pheromone on edge $(i,j)$ and $\eta_{ij} = 1/d_{ij}$ is the heuristic (visibility). After all ants finish, pheromone is updated globally:

$$\tau_{ij} \leftarrow (1-\rho)\, \tau_{ij} + \sum_{k=1}^{m} \Delta \tau_{ij}^{k}, \quad \Delta \tau_{ij}^{k} = \begin{cases} Q / L_k & \text{if ant } k \text{ used edge } (i,j) \\ 0 & \text{otherwise} \end{cases}$$

Parameters: $\alpha$ (pheromone weight), $\beta$ (heuristic weight), $\rho$ (evaporation rate), $Q$ (deposit constant), $m$ (number of ants), and the number of iterations.

In [1]:
# --- TSP utilities (same format as used in lab 3 / lab 5) ---
import math
import random
import time
from statistics import mean


def read_tsp(file_path):
    """Parse a TSPLIB EUC_2D instance. Returns {name, dimension, coords (0-indexed list)}."""
    name = None
    dimension = None
    in_coord_section = False
    coords_by_id = {}
    with open(file_path, "r") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith("NAME"):
                name = line.split(":", 1)[1].strip() if ":" in line else line.split()[-1]
            elif line.startswith("DIMENSION"):
                rhs = line.split(":", 1)[1].strip() if ":" in line else line.split()[-1]
                dimension = int(rhs)
            elif line == "NODE_COORD_SECTION":
                in_coord_section = True
            elif line == "EOF":
                break
            elif in_coord_section:
                parts = line.split()
                if len(parts) >= 3:
                    coords_by_id[int(parts[0])] = (float(parts[1]), float(parts[2]))
    coords = [coords_by_id[k] for k in sorted(coords_by_id)]
    if dimension is None:
        dimension = len(coords)
    return {"name": name or file_path, "dimension": dimension, "coords": coords}


def build_distance_matrix(coords):
    """Pre-compute the Euclidean distance matrix. Cheaper than recomputing inside the hot loop."""
    n = len(coords)
    D = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            d = math.hypot(coords[i][0] - coords[j][0], coords[i][1] - coords[j][1])
            D[i][j] = D[j][i] = d
    return D


def tour_length(tour, D):
    """Length of a closed tour (returns to start)."""
    total = 0.0
    n = len(tour)
    for i in range(n):
        total += D[tour[i]][tour[(i + 1) % n]]
    return total


def nearest_neighbor_tour(D, start=0):
    """Greedy NN tour. Used only for pheromone initialization (tau_0 = m / L_nn)."""
    n = len(D)
    unvisited = set(range(n)) - {start}
    tour = [start]
    current = start
    while unvisited:
        nxt = min(unvisited, key=lambda j: D[current][j])
        tour.append(nxt)
        unvisited.remove(nxt)
        current = nxt
    return tour

In [2]:
# --- Ant System core ---

def construct_ant_tour(D, pheromone, alpha, beta, start=None):
    """
    Build one complete tour for a single ant.
    At each step, pick the next unvisited city with probability
        p_ij ~ tau_ij^alpha * eta_ij^beta,  where  eta_ij = 1 / d_ij.
    Roulette-wheel sampling from that distribution.
    """
    n = len(D)
    if start is None:
        start = random.randrange(n)
    tour = [start]
    visited = [False] * n
    visited[start] = True
    current = start

    for _ in range(n - 1):
        # Score every unvisited city
        weights = []
        candidates = []
        for j in range(n):
            if visited[j]:
                continue
            tau = pheromone[current][j]
            eta = 1.0 / D[current][j] if D[current][j] > 0 else 1e12
            weights.append((tau ** alpha) * (eta ** beta))
            candidates.append(j)

        total = sum(weights)
        if total <= 0.0:
            # Numerical underflow fallback: uniform choice among unvisited
            nxt = random.choice(candidates)
        else:
            r = random.random() * total
            cum = 0.0
            nxt = candidates[-1]
            for j, w in zip(candidates, weights):
                cum += w
                if r <= cum:
                    nxt = j
                    break

        tour.append(nxt)
        visited[nxt] = True
        current = nxt

    return tour


def update_pheromone(pheromone, tours, lengths, rho, Q):
    """
    Global pheromone update for Ant System:
        (1) evaporate:   tau <- (1 - rho) * tau
        (2) deposit:     tau_ij += sum_k (Q / L_k)  for each edge (i,j) used by ant k
    Updates are applied symmetrically on the undirected TSP graph.
    """
    n = len(pheromone)
    for i in range(n):
        row = pheromone[i]
        for j in range(n):
            row[j] *= (1.0 - rho)

    for tour, L in zip(tours, lengths):
        deposit = Q / L if L > 0 else 0.0
        for k in range(len(tour)):
            a = tour[k]
            b = tour[(k + 1) % len(tour)]
            pheromone[a][b] += deposit
            pheromone[b][a] += deposit  # symmetric TSP


def ant_system(coords, n_ants=10, iterations=10,
               alpha=1.0, beta=2.0, rho=0.5, Q=1.0,
               seed=None, record_history=False):
    """
    Run the Ant System algorithm on a TSP instance (given as coordinates).
    Returns (best_tour, best_length, history)
    where history is a list of (iteration, iter_best_length, global_best_length)
    if record_history is True, else None.

    Pheromone initialization: tau_0 = n_ants / L_nn  (Dorigo's rule of thumb)
    where L_nn is the length of the nearest-neighbor tour from city 0.
    """
    if seed is not None:
        random.seed(seed)

    D = build_distance_matrix(coords)
    n = len(D)

    L_nn = tour_length(nearest_neighbor_tour(D, start=0), D)
    tau_0 = n_ants / L_nn
    pheromone = [[tau_0] * n for _ in range(n)]

    best_tour = None
    best_length = float("inf")
    history = [] if record_history else None

    for it in range(iterations):
        tours = [construct_ant_tour(D, pheromone, alpha, beta) for _ in range(n_ants)]
        lengths = [tour_length(t, D) for t in tours]

        iter_best_idx = min(range(n_ants), key=lambda k: lengths[k])
        iter_best_length = lengths[iter_best_idx]
        if iter_best_length < best_length:
            best_length = iter_best_length
            best_tour = tours[iter_best_idx][:]

        update_pheromone(pheromone, tours, lengths, rho, Q)

        if record_history:
            history.append((it + 1, iter_best_length, best_length))

    return best_tour, best_length, history

### Task (a) - One iteration with 10 ants

Instance: `berlin52` (52 cities, optimum = 7542). We do exactly **one** AS iteration
with **10 ants**, print every ant's tour length, and mark the iteration-best.

In [3]:
random.seed(1)  # reproducible demo

tsp = read_tsp("../data/A/berlin52.tsp")
coords = tsp["coords"]
D = build_distance_matrix(coords)
n = len(coords)

L_nn = tour_length(nearest_neighbor_tour(D, start=0), D)
print(f"Instance: {tsp['name']}  (n={n}, known optimum = 7542)")
print(f"Nearest-neighbor baseline (L_nn) = {L_nn:.2f}\n")

# Parameters for a single iteration with 10 ants
N_ANTS = 10
ALPHA  = 1.0
BETA   = 2.0
RHO    = 0.5
Q      = 1.0

# Initialize pheromone uniformly to tau_0 = n_ants / L_nn
tau_0 = N_ANTS / L_nn
pheromone = [[tau_0] * n for _ in range(n)]
print(f"Initial pheromone (uniform): tau_0 = n_ants / L_nn = {tau_0:.6e}\n")

# Each ant builds one tour
tours = [construct_ant_tour(D, pheromone, ALPHA, BETA) for _ in range(N_ANTS)]
lengths = [tour_length(t, D) for t in tours]

best_k = min(range(N_ANTS), key=lambda k: lengths[k])

print("Ant  Length      vs L_nn   First 10 cities of the tour")
print("-" * 70)
for k in range(N_ANTS):
    marker = "  <-- iteration best" if k == best_k else ""
    rel = 100.0 * (lengths[k] - L_nn) / L_nn
    print(f"{k+1:>3}  {lengths[k]:>9.2f}  {rel:>+6.2f}%   {tours[k][:10]}{marker}")

print(f"\nIteration best ant = #{best_k+1} with length {lengths[best_k]:.2f}")
print(f"Mean of 10 ants    = {mean(lengths):.2f}")

# Now apply the pheromone update (evaporate + deposit) and show how some
# pheromone values changed so the effect is visible.
update_pheromone(pheromone, tours, lengths, RHO, Q)

# Sample a few edges on the best-ant tour and show their new pheromone level
print("\nPheromone on 5 edges of the iteration-best tour after update:")
print(f"(initial tau_0 = {tau_0:.6e})")
for i in range(5):
    a, b = tours[best_k][i], tours[best_k][i+1]
    print(f"  edge ({a:>3},{b:>3})  d={D[a][b]:>7.2f}   tau = {pheromone[a][b]:.6e}")

Instance: berlin52  (n=52, known optimum = 7542)
Nearest-neighbor baseline (L_nn) = 8980.92

Initial pheromone (uniform): tau_0 = n_ants / L_nn = 1.113472e-03

Ant  Length      vs L_nn   First 10 cities of the tour
----------------------------------------------------------------------
  1   16388.59  +82.48%   [8, 9, 40, 6, 1, 41, 22, 20, 16, 21]
  2   17972.91  +100.12%   [31, 48, 33, 35, 15, 20, 36, 37, 5, 14]
  3   18481.50  +105.79%   [32, 21, 38, 35, 0, 34, 33, 36, 39, 37]
  4   14904.02  +65.95%   [32, 37, 36, 23, 47, 38, 35, 34, 33, 29]
  5   16154.87  +79.88%   [50, 9, 8, 7, 17, 21, 30, 48, 38, 35]
  6   15570.23  +73.37%   [0, 18, 7, 42, 3, 5, 47, 23, 4, 14]
  7   17306.59  +92.70%   [19, 9, 8, 40, 18, 44, 14, 4, 34, 35]
  8   14226.09  +58.40%   [15, 22, 2, 48, 35, 34, 33, 43, 31, 19]  <-- iteration best
  9   15323.52  +70.62%   [35, 34, 33, 39, 37, 47, 24, 46, 25, 12]
 10   15362.70  +71.06%   [12, 26, 27, 3, 47, 23, 4, 5, 14, 33]

Iteration best ant = #8 with length 14226.

### Task (b) - 10 iterations, track the global best

Same parameters as above, now for 10 iterations. For each iteration we log
the iteration-best length and the running global best.

In [4]:
best_tour, best_length, history = ant_system(
    coords,
    n_ants=N_ANTS,
    iterations=10,
    alpha=ALPHA, beta=BETA, rho=RHO, Q=Q,
    seed=1,
    record_history=True,
)

print("Iter   iter-best      global-best")
print("-" * 40)
for it, ib, gb in history:
    print(f"{it:>4}   {ib:>10.2f}     {gb:>10.2f}")

gap = 100.0 * (best_length - 7542) / 7542
print(f"\nGlobal best after 10 iterations: {best_length:.2f}  (gap to optimum 7542: {gap:+.2f}%)")
print(f"Best tour (first 10 cities): {best_tour[:10]}...")
print(f"NN baseline was {L_nn:.2f} -> AS improvement over NN: {100.0*(L_nn - best_length)/L_nn:+.2f}%")

Iter   iter-best      global-best
----------------------------------------
   1     14226.09       14226.09
   2     13516.05       13516.05
   3     13997.38       13516.05
   4     11834.73       11834.73
   5     11677.65       11677.65
   6      9735.09        9735.09
   7      9684.06        9684.06
   8      9732.36        9684.06
   9      9788.66        9684.06
  10      9189.79        9189.79

Global best after 10 iterations: 9189.79  (gap to optimum 7542: +21.85%)
Best tour (first 10 cities): [7, 40, 44, 18, 31, 48, 0, 21, 17, 30]...
NN baseline was 8980.92 -> AS improvement over NN: -2.33%


### Parameter-settings test

Several AS configurations run on `berlin52`, each repeated 3 times with
different seeds. We vary $\alpha, \beta, \rho$, number of ants, and iterations.
Average / best / worst tour length across runs is reported.

Results are also written to `results_as_tsp.txt` in a human-readable format.

In [5]:
param_settings = [
    # (tag,       alpha, beta, rho, n_ants, iterations)
    ("baseline",    1.0,  2.0, 0.5, 10, 30),
    ("high-beta",   1.0,  5.0, 0.5, 10, 30),   # heuristic-dominated (more greedy)
    ("high-alpha",  2.0,  2.0, 0.5, 10, 30),   # pheromone-dominated
    ("low-evap",    1.0,  2.0, 0.1, 10, 30),   # pheromone persists longer
    ("high-evap",   1.0,  2.0, 0.9, 10, 30),   # pheromone forgotten quickly
    ("more-ants",   1.0,  2.0, 0.5, 30, 30),   # bigger colony
    ("more-iters",  1.0,  2.0, 0.5, 10, 100),  # longer run
]

NUM_RUNS = 3
KNOWN_OPT = 7542  # berlin52 optimum

results = []
print(f"Instance: {tsp['name']}  (n={n}, known optimum = {KNOWN_OPT})")
print(f"NN baseline = {L_nn:.2f}\n")
print(f"{'Setting':<12} {'alpha':>5} {'beta':>5} {'rho':>5} {'ants':>5} {'iter':>5} "
      f"{'avg':>10} {'best':>10} {'worst':>10} {'gap%':>7} {'t[s]':>7}")
print("-" * 92)

for tag, alpha, beta, rho, n_ants, iterations in param_settings:
    run_lengths = []
    t0 = time.time()
    for r in range(NUM_RUNS):
        _, L, _ = ant_system(
            coords, n_ants=n_ants, iterations=iterations,
            alpha=alpha, beta=beta, rho=rho, Q=1.0,
            seed=1000 + r,
        )
        run_lengths.append(L)
    elapsed = time.time() - t0

    avg_L   = mean(run_lengths)
    best_L  = min(run_lengths)
    worst_L = max(run_lengths)
    gap = 100.0 * (best_L - KNOWN_OPT) / KNOWN_OPT

    print(f"{tag:<12} {alpha:>5.1f} {beta:>5.1f} {rho:>5.2f} {n_ants:>5} {iterations:>5} "
          f"{avg_L:>10.2f} {best_L:>10.2f} {worst_L:>10.2f} {gap:>+6.2f}% {elapsed:>6.2f}")

    results.append({
        "setting": tag, "alpha": alpha, "beta": beta, "rho": rho,
        "n_ants": n_ants, "iterations": iterations,
        "runs": NUM_RUNS,
        "all_runs": run_lengths,
        "avg": avg_L, "best": best_L, "worst": worst_L,
        "gap_pct_best_vs_opt": gap,
        "total_time_s": elapsed,
    })

# --- Write human-readable text report ---
with open("results_as_tsp.txt", "w") as f:
    f.write("=" * 78 + "\n")
    f.write("  ANT SYSTEM (AS) -- Lab 8 parameter test\n")
    f.write("  TSP instance: " + tsp["name"] + f"  (n={n}, known optimum = {KNOWN_OPT})\n")
    f.write("=" * 78 + "\n\n")
    f.write(f"Nearest-neighbor baseline : {L_nn:.2f}\n")
    f.write(f"Runs per setting          : {NUM_RUNS}\n")
    f.write(f"Seeds                     : 1000..{1000 + NUM_RUNS - 1}\n")
    f.write(f"Pheromone init            : tau_0 = n_ants / L_nn\n\n")

    f.write(f"{'Setting':<12} {'alpha':>5} {'beta':>5} {'rho':>5} {'ants':>5} {'iter':>5} "
            f"{'avg':>10} {'best':>10} {'worst':>10} {'gap%':>7}\n")
    for r in results:
        f.write(f"{r['setting']:<12} {r['alpha']:>5.1f} {r['beta']:>5.1f} {r['rho']:>5.2f} "
                f"{r['n_ants']:>5} {r['iterations']:>5} {r['avg']:>10.2f} "
                f"{r['best']:>10.2f} {r['worst']:>10.2f} {r['gap_pct_best_vs_opt']:>+6.2f}%\n")
    f.write("\n")

    for r in results:
        f.write("-" * 78 + "\n")
        f.write(f"[{r['setting']}]\n")
        f.write(f"  parameters : alpha={r['alpha']}, beta={r['beta']}, rho={r['rho']}, "
                f"n_ants={r['n_ants']}, iterations={r['iterations']}, Q=1.0\n")
        f.write(f"  runs       : {r['runs']}\n")
        f.write(f"  individual : " + ", ".join(f"{L:.2f}" for L in r['all_runs']) + "\n")
        f.write(f"  avg={r['avg']:.2f}  best={r['best']:.2f}  worst={r['worst']:.2f}  "
                f"gap_to_opt={r['gap_pct_best_vs_opt']:+.2f}%  total_time={r['total_time_s']:.2f}s\n\n")

    f.write("=" * 78 + "\n")
    f.write("End of report\n")

print("\nResults saved to 'results_as_tsp.txt'")

Instance: berlin52  (n=52, known optimum = 7542)
NN baseline = 8980.92

Setting      alpha  beta   rho  ants  iter        avg       best      worst    gap%    t[s]
--------------------------------------------------------------------------------------------
baseline       1.0   2.0  0.50    10    30    8256.30    8054.08    8517.81  +6.79%   0.64
high-beta      1.0   5.0  0.50    10    30    8042.64    7962.04    8095.23  +5.57%   0.64
high-alpha     2.0   2.0  0.50    10    30    8428.08    8116.14    8792.89  +7.61%   0.61
low-evap       1.0   2.0  0.10    10    30    9241.30    9182.69    9324.03 +21.75%   0.61
high-evap      1.0   2.0  0.90    10    30    8194.23    7709.70    8510.37  +2.22%   0.61
more-ants      1.0   2.0  0.50    30    30    8023.19    7911.52    8171.04  +4.90%   1.88
more-iters     1.0   2.0  0.50    10   100    7912.75    7829.83    8052.24  +3.82%   2.33

Results saved to 'results_as_tsp.txt'


## Assignment A8: Ant Colony System (ACS) for TSP

**Ant Colony System (ACS)** introduces three major changes compared to the Ant System (AS):
1. **Pseudorandom Proportional Rule**: To choose the next city, an ant heavily exploits the best available choice with a probability $q_0$, and explores using the roulette wheel (like AS) with probability $1 - q_0$.
2. **Local Pheromone Update**: As ants visit edges, they immediately reduce the pheromone level on them, encouraging subsequent ants to explore different paths.
3. **Global Pheromone Update**: At the end of an iteration, only the **global best ant** deposits pheromone, highly emphasizing the best-found tour.

We will run ACS on two instances: `pr107.tsp` and `lu980.tsp` and test different parameters.

In [6]:
# --- Ant Colony System (ACS) core ---

def construct_acs_tour(D, pheromone, beta, q0, xi, tau0, start=None):
    """
    Build one complete tour for a single ant using ACS rules.
    This also applies the local pheromone update immediately after traversing an edge.
    """
    n = len(D)
    if start is None:
        start = random.randrange(n)
        
    tour = [start]
    visited = [False] * n
    visited[start] = True
    current = start

    for _ in range(n - 1):
        best_val = -1.0
        best_nxt = -1
        
        weights = []
        candidates = []
        
        # Calculate attractiveness for unvisited cities
        for j in range(n):
            if visited[j]:
                continue
            
            tau = pheromone[current][j]
            eta = 1.0 / D[current][j] if D[current][j] > 0 else 1e12
            attr = tau * (eta ** beta)
            
            if attr > best_val:
                best_val = attr
                best_nxt = j
                
            weights.append(attr)
            candidates.append(j)

        # Pseudorandom proportional rule
        if random.random() <= q0:
            nxt = best_nxt  # Exploit
        else:
            # Explore (Roulette Wheel)
            total = sum(weights)
            if total <= 0.0:
                nxt = random.choice(candidates)
            else:
                r = random.random() * total
                cum = 0.0
                nxt = candidates[-1]
                for j, w in zip(candidates, weights):
                    cum += w
                    if r <= cum:
                        nxt = j
                        break

        # Move to next
        tour.append(nxt)
        visited[nxt] = True
        
        # Local Pheromone Update
        # tau_ij = (1 - xi) * tau_ij + xi * tau0
        pheromone[current][nxt] = (1.0 - xi) * pheromone[current][nxt] + xi * tau0
        pheromone[nxt][current] = pheromone[current][nxt] # Symmetric
        
        current = nxt
        
    # Local update on the returning edge
    last, first = tour[-1], tour[0]
    pheromone[last][first] = (1.0 - xi) * pheromone[last][first] + xi * tau0
    pheromone[first][last] = pheromone[last][first]

    return tour


def global_update_acs(pheromone, best_tour, best_length, rho):
    """
    Global update is applied ONLY to the edges of the globally best tour.
    tau_ij = (1 - rho) * tau_ij + rho * (1 / L_best)
    """
    n = len(best_tour)
    delta_tau = 1.0 / best_length if best_length > 0 else 0.0
    
    # We do not evaporate non-best edges in canonical ACS globally! 
    # (Evaporation for non-best edges happens via local update).
    # Some variants evaporate everything, but standard ACS only updates best tour.
    # Actually, Dorigo's typical ACS is: 
    # tau_ij <- (1 - rho) * tau_ij + rho / L_gb  FOR (i,j) in T_gb
    
    for k in range(n):
        a = best_tour[k]
        b = best_tour[(k + 1) % n]
        
        pheromone[a][b] = (1.0 - rho) * pheromone[a][b] + rho * delta_tau
        pheromone[b][a] = pheromone[a][b]


def acs_system(coords, n_ants=10, iterations=10, 
               beta=2.0, q0=0.9, rho=0.1, xi=0.1, 
               seed=None, record_history=False):
    """
    Run Ant Colony System (ACS).
    """
    if seed is not None:
        random.seed(seed)

    D = build_distance_matrix(coords)
    n = len(D)

    # Initial pheromone tau_0 = 1 / (n * L_nn)  (standard for ACS)
    L_nn = tour_length(nearest_neighbor_tour(D, start=0), D)
    tau_0 = 1.0 / (n * L_nn) if L_nn > 0 else 1e-12
    
    pheromone = [[tau_0] * n for _ in range(n)]

    global_best_tour = None
    global_best_length = float("inf")
    history = [] if record_history else None

    for it in range(iterations):
        tours = []
        lengths = []
        
        # Note: Canonical ACS lets all ants build tours in parallel (step-by-step)
        # to ensure local updates are "felt" immediately by others. 
        # For simplicity and speed in basic Python, evaluating sequentially 
        # is a widespread, acceptable approximation of the parallel process.
        for _ in range(n_ants):
            t = construct_acs_tour(D, pheromone, beta, q0, xi, tau_0)
            L = tour_length(t, D)
            tours.append(t)
            lengths.append(L)

        iter_best_idx = min(range(n_ants), key=lambda k: lengths[k])
        iter_best_length = lengths[iter_best_idx]
        
        if iter_best_length < global_best_length:
            global_best_length = iter_best_length
            global_best_tour = tours[iter_best_idx][:]

        # Global update on global best tour only
        global_update_acs(pheromone, global_best_tour, global_best_length, rho)

        if record_history:
            history.append((it + 1, iter_best_length, global_best_length))

    return global_best_tour, global_best_length, history


### Running experiments for A8

Testing the implemented **ACS** on the specific instances:
- `pr107.tsp` ($n=107$, optimum is known to be around $44303$)
- `lu980.tsp` ($n=980$, optimum is known to be around $11340$)

We will test multiple settings to measure their effect on the final solution quality.

In [7]:
import os

# Instances
instances_info = [
    {"path": "../data/A/pr107.tsp", "opt": 44303},
    {"path": "../data/B/lu980.tsp", "opt": 11340}
]

acs_settings = [
    # (tag, beta, q0, rho, xi, n_ants, iterations)
    ("acs_base", 2.0, 0.9, 0.1, 0.1, 10, 20),
    ("high_q0",  2.0, 0.95,0.1, 0.1, 10, 20),  # More greedy exploitation
    ("low_q0",   2.0, 0.5, 0.1, 0.1, 10, 20),  # Less greedy, more exploration
    ("high_beta",5.0, 0.9, 0.1, 0.1, 10, 20)   # Higher weight to heuristics
]

NUM_RUNS = 2  # To keep computation time reasonable

acs_results = []

for inst in instances_info:
    filepath = inst["path"]
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}. Skipping.")
        continue
        
    tsp_data = read_tsp(filepath)
    coords = tsp_data["coords"]
    n_cities = tsp_data["dimension"]
    opt_val = inst["opt"]
    
    print("=" * 80)
    print(f"Running ACS on {tsp_data['name']} (n={n_cities}, opt≈{opt_val})")
    print(f"{'Setting':<10} {'beta':>4} {'q0':>4} {'rho':>4} {'xi':>4} {'avg':>10} {'best':>10} {'gap%':>7} {'t[s]':>6}")
    print("-" * 80)
    
    for tag, beta, q0, rho, xi, n_ants, iterations in acs_settings:
        run_lengths = []
        t0 = time.time()
        for r in range(NUM_RUNS):
            _, L, _ = acs_system(
                coords, n_ants=n_ants, iterations=iterations,
                beta=beta, q0=q0, rho=rho, xi=xi,
                seed=1000 + r
            )
            run_lengths.append(L)
            
        elapsed = time.time() - t0
        avg_L = mean(run_lengths)
        best_L = min(run_lengths)
        gap = 100.0 * (best_L - opt_val) / opt_val

        col = f"{tag:<10} {beta:>4.1f} {q0:>4.2f} {rho:>4.2f} {xi:>4.2f} {avg_L:>10.2f} {best_L:>10.2f} {gap:>+6.2f}% {elapsed:>6.1f}"
        print(col)
        
        acs_results.append({
            "instance": tsp_data['name'], "n": n_cities, "opt": opt_val,
            "setting": tag, "beta": beta, "q0": q0, "rho": rho, "xi": xi,
            "n_ants": n_ants, "iterations": iterations,
            "runs": NUM_RUNS, "all_runs": run_lengths,
            "avg": avg_L, "best": best_L,
            "gap_pct_best_vs_opt": gap, "total_time_s": elapsed
        })


# --- Write human-readable text report for assignment A8 ---
outFile = "results_acs_tsp.txt"
with open(outFile, "w") as f:
    f.write("=" * 80 + "\n")
    f.write("  ANT COLONY SYSTEM (ACS) -- Assignment A8 Experimental Results\n")
    f.write("=" * 80 + "\n\n")
    for r in acs_results:
        f.write("-" * 80 + "\n")
        f.write(f"Instance   : {r['instance']} (n={r['n']}, known opt={r['opt']})\n")
        f.write(f"Setting tag: {r['setting']}\n")
        f.write(f"Parameters : beta={r['beta']}, q0={r['q0']}, rho={r['rho']}, xi={r['xi']}, n_ants={r['n_ants']}, iterations={r['iterations']}\n")
        f.write(f"Runs       : {r['runs']}\n")
        f.write(f"Individual : " + ", ".join(f"{L:.2f}" for L in r['all_runs']) + "\n")
        f.write(f"Metrics    : avg={r['avg']:.2f} | best={r['best']:.2f} | gap_to_opt={r['gap_pct_best_vs_opt']:+.2f}% | time_total={r['total_time_s']:.1f}s\n\n")

print(f"\nACS experimental results saved to '{outFile}'")


Running ACS on pr107 (n=107, opt≈44303)
Setting    beta   q0  rho   xi        avg       best    gap%   t[s]
--------------------------------------------------------------------------------
acs_base    2.0 0.90 0.10 0.10   47184.17   46733.20  +5.49%    1.0
high_q0     2.0 0.95 0.10 0.10   46472.37   46273.12  +4.45%    0.8
low_q0      2.0 0.50 0.10 0.10   55351.30   52972.04 +19.57%    0.9
high_beta   5.0 0.90 0.10 0.10   46098.73   45736.09  +3.23%    0.9
Running ACS on lu980 (n=980, opt≈11340)
Setting    beta   q0  rho   xi        avg       best    gap%   t[s]
--------------------------------------------------------------------------------
acs_base    2.0 0.90 0.10 0.10   15563.98   15378.53 +35.61%   88.7
high_q0     2.0 0.95 0.10 0.10   14739.93   14728.58 +29.88%   47.8
low_q0      2.0 0.50 0.10 0.10   23414.88   21726.67 +91.59%   50.3
high_beta   5.0 0.90 0.10 0.10   13242.24   13116.36 +15.66%   48.7

ACS experimental results saved to 'results_acs_tsp.txt'


## Ant Colony Optimization - Assignment A8 Report

### 1. Problem Definition
The **Traveling Salesman Problem (TSP)** is a classic algorithmic problem in the field of computer science and operations research. It focuses on optimization: given a list of cities and the distances between each pair of cities, the goal is to find the shortest possible route that visits each city exactly once and returns to the origin city. This problem belongs to the class of NP-hard problems. We use heuristic and metaheuristic approaches like Ant Colony Optimization (ACO) to find near-optimal solutions.

The experiments described in this report involve instances taken from TSPLIB:
- `berlin52`: 52 cities, optimal tour length = 7542 (used in lab session AS tests).
- `pr107`: 107 cities, optimal tour length = 44303.
- `lu980`: 980 cities, optimal tour length = 11340.

### 2. Algorithms Used

#### 2.1 Ant System (AS)
Ant System mimics the foraging behavior of natural ants.
**Steps:**
1. Initialize pheromones $\tau_{ij}$ on all edges (to $\tau_0 = m/L_{nn}$).
2. For a fixed number of iterations:
   - $m$ ants construct their tours. At city $i$, the next city $j$ is chosen probabilistically:
     $$p_{ij} = \frac{[\tau_{ij}]^\alpha \, [\eta_{ij}]^\beta}{\sum_{k} [\tau_{ik}]^\alpha \, [\eta_{ik}]^\beta}$$
   - Calculate the tour length $L_k$ for all ants.
   - **Global Pheromone Update:** $\tau_{ij} \leftarrow (1-\rho)\tau_{ij}$. Each ant deposits pheromone: $\Delta\tau^k_{ij} = Q/L_k$.
3. Return the global best tour found.

#### 2.2 Ant Colony System (ACS)
Ant Colony System introduces a few performance-improving modifiers over the basic AS, focusing on stronger exploitation of the search experience.
**Steps:**
1. Initialize pheromones $\tau_{ij}$ to $\tau_0 = 1 / (n \cdot L_{nn})$.
2. For a fixed number of iterations:
   - For each city step across all $m$ ants:
     - **Pseudorandom Proportional Rule:** generate random $q \in [0, 1]$. If $q \le q_0$, the ant chooses the next city that maximizes $\tau_{ij} \cdot \eta_{ij}^\beta$ (pure exploitation). Otherwise, it explores probabilistically using the roulette wheel selection as in AS.
     - **Local Pheromone Update:** $\tau_{ij} = (1 - \xi)\tau_{ij} + \xi\tau_0$. This prevents subsequent ants from deterministically following the exact same path.
   - **Global Pheromone Update:** At the end of the iteration, evaporation and deposition are applied **only** to the edges of the globally best path found so far: $\tau_{ij} \leftarrow (1-\rho)\tau_{ij} + \rho(1/L_{best})$.

### 3. Parameter Settings & Experiments

**AS Parameters (Tested on berlin52)**
- **baseline:** $\alpha=1.0$, $\beta=2.0$, $\rho=0.50$, $10$ ants, $30$ iterations
- **high-beta:** $\beta=5.0$, **high-alpha:** $\alpha=2.0$
- **low-evap:** $\rho=0.10$, **high-evap:** $\rho=0.90$
- **more-ants:** $30$ ants, **more-iters:** $100$ iterations

**ACS Parameters (Tested on pr107 & lu980)**
Simulated with $10$ ants over $20$ iterations:
- **acs_base:** $q_0=0.9$, $\beta=2.0$, $\rho=0.1$, $\xi=0.1$
- **high_q0:** $q_0=0.95$ (Stronger exploitation)
- **low_q0:** $q_0=0.5$ (More exploration)
- **high_beta:** $\beta=5.0$ (Heuristic dominance)

### 4. Comparative Results

#### 4.1 AS on `berlin52` (Opt = 7542)
| Setting      | Avg Length | Best Length | Gap to Opt | Time (s)|
|--------------|------------|-------------|------------|---------|
| baseline     | 8256.30    | 8054.08     | +6.79%     | 0.64    |
| high-beta    | 8042.64    | 7962.04     | +5.57%     | 0.64    |
| high-alpha   | 8428.08    | 8116.14     | +7.61%     | 0.61    |
| low-evap     | 9241.30    | 9182.69     | +21.75%    | 0.61    |
| high-evap    | 8194.23    | 7709.70     | +2.22%     | 0.61    |
| more-ants    | 8023.19    | 7911.52     | +4.90%     | 1.88    |
| more-iters   | 7912.75    | 7829.83     | +3.82%     | 2.33    |

#### 4.2 ACS on `pr107` (Opt = 44303)
| Setting      | Avg Length | Best Length | Gap to Opt | Time (s)|
|--------------|------------|-------------|------------|---------|
| acs_base     | 47184.17   | 46733.20    | +5.49%     | 1.0     |
| high_q0      | 46472.37   | 46273.12    | +4.45%     | 0.8     |
| low_q0       | 55351.30   | 52972.04    | +19.57%    | 0.9     |
| high_beta    | 46098.73   | 45736.09    | +3.23%     | 0.9     |

#### 4.3 ACS on `lu980` (Opt = 11340)
| Setting      | Avg Length | Best Length | Gap to Opt | Time (s)|
|--------------|------------|-------------|------------|---------|
| acs_base     | 15563.98   | 15378.53    | +35.61%    | 88.7    |
| high_q0      | 14739.93   | 14728.58    | +29.88%    | 47.8    |
| low_q0       | 23414.88   | 21726.67    | +91.59%    | 50.3    |
| high_beta    | 13242.24   | 13116.36    | +15.66%    | 48.7    |

### 5. Discussion of Results

#### Observations from AS (Ant System)
1. **Evaporation Rate ($\rho$):** A high evaporation rate (`high-evap`, $\rho=0.9$) resulted in the closest gap to optimal (2.22%). Rapid pheromone decay keeps the search focused greedily around the currently discovered best edges. Conversely, low evaporation (`low-evap`) yielded completely unoptimized solutions (+21.75% gap) because bad early decisions persisted across the pheromone matrix.
2. **Computational Effort:** Adding more iterations (`more-iters`) yielded better results (+3.82% gap) at the expense of proportional run time.

#### Observations from ACS (Ant Colony System)
1. **Balance of Exploitation (`q0`):** Decreasing $q_0$ to $0.5$ forces ACS to act largely like the standard AS, exploring broadly. In both tested datasets, this resulted in completely derailed routes (19.57% gap for `pr107` and 91.59% gap for `lu980`). Increasing the greediness/exploitation factor to $0.95$ (`high_q0`) significantly snapped the routes closer to the optimum.
2. **Beta Dependence ($\beta$):** The best overall parameters tested for both `pr107` and `lu980` involved setting $\beta=5.0$ (`high_beta`). Valuing the heuristic (inverse distance) heavily is crucial to avoiding long jumps across the board, achieving a 3.23% gap in `pr107` and an impressive 15.66% gap in the very complex 980-node problem (considering the extremely constrained iteration count).
3. **Complexity Limitations:** Standard baseline configurations generated decent results for $n=107$, but logically struggled to find near-optimal lines purely via 20 iterations scaling up to the $n=980$ node system (+35% gap vs 11340). To reach tighter margins on large instances, more iterations and larger ant cohorts are strictly necessary.